In [ ]:
# generar_todas_las_redes.py

import csv
from itertools import combinations
from collections import Counter, defaultdict
from unidecode import unidecode
from thefuzz import fuzz




Sección 1 desambiaguación de autores 

In [5]:


def crear_forma_base(nombre):
    """Crea una versión "base" y súper limpia de un nombre."""
    nombre_limpio = nombre.lower()
    reemplazos = {'a´': 'á', 'e´': 'é', 'i´': 'í', 'o´': 'ó', 'u´': 'ú', 'n~': 'ñ'}
    for erroneo, correcto in reemplazos.items():
        nombre_limpio = nombre_limpio.replace(erroneo, correcto)
    nombre_limpio = unidecode(nombre_limpio)
    nombre_limpio = ''.join(c for c in nombre_limpio if c.isalpha() or c.isspace())
    titulos = ["dr", "dra", "msc", "lic", "prof"]
    partes = nombre_limpio.split()
    partes_sin_titulos = [p for p in partes if p not in titulos]
    return " ".join(partes_sin_titulos)

def desambiguar_con_similitud(lista_autores_brutos, umbral=90):
    """Agrupa autores usando un umbral de similitud en sus formas base."""
    print("Iniciando desambiguación de autores con similitud de strings...")
    formas_base = {original: crear_forma_base(original) for original in lista_autores_brutos}
    grupos = [[autor] for autor in lista_autores_brutos]
    hubo_fusion = True
    while hubo_fusion:
        hubo_fusion = False
        i = 0
        while i < len(grupos):
            j = i + 1
            while j < len(grupos):
                autor1, autor2 = grupos[i][0], grupos[j][0]
                base1, base2 = formas_base[autor1], formas_base[autor2]
                if fuzz.token_set_ratio(base1, base2) > umbral:
                    grupos[i].extend(grupos[j])
                    del grupos[j]
                    hubo_fusion = True
                else: j += 1
            i += 1
            
    mapa_desambiguacion = {}
    for grupo in grupos:
        nombre_canonico_bruto = max(grupo, key=len)
        reemplazos_tildes = {'a´': 'á', 'e´': 'é', 'i´': 'í', 'o´': 'ó', 'u´': 'ú', 'n~': 'ñ'}
        nombre_canonico_limpio = nombre_canonico_bruto
        for erroneo, correcto in reemplazos_tildes.items():
            nombre_canonico_limpio = nombre_canonico_limpio.replace(erroneo, correcto).replace(erroneo.upper(), correcto.upper())
        nombre_canonico_limpio = nombre_canonico_limpio.replace("JuliÆn", "Julián").replace("Elisaˆngela", "Elisângela")
        for variante in grupo:
            mapa_desambiguacion[variante] = nombre_canonico_limpio
            
    n_variantes, n_canonicos = len(lista_autores_brutos), len(set(mapa_desambiguacion.values()))
    print(f"-> {n_variantes} variantes de nombres agrupadas en {n_canonicos} autores únicos.")
    return mapa_desambiguacion


SECCIÓN 2: LECTURA Y PREPROCESAMIENTO DE DATOS

In [6]:


archivo_entrada_csv = 'extraccion_completa_fireworks.csv'
articulos = []
todos_los_autores_brutos = set()

print(f"\nLeyendo datos desde '{archivo_entrada_csv}'...")
try:
    with open(archivo_entrada_csv, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            # Limpiamos y preparamos cada campo
            row['autores'] = [a.strip() for a in row['autores'].split('|') if a.strip()]
            row['afiliaciones'] = list(set([a.strip() for a in row['afiliaciones'].split('|') if a.strip()]))
            row['palabras_clave'] = list(set([k.strip().lower() for k in row['palabras_clave'].split('|') if k.strip()]))
            row['tematica'] = row['tematica'].strip()

            if row['autores']:
                articulos.append(row)
                for autor in row['autores']:
                    todos_los_autores_brutos.add(autor)
except FileNotFoundError:
    print(f"!!! ERROR: No se encontró el archivo '{archivo_entrada_csv}'.")
    exit()

# Creamos el mapa de desambiguación que usaremos en todas las redes
mapa_nombres = desambiguar_con_similitud(list(todos_los_autores_brutos))
autores_canonicos_nodos = sorted(list(set(mapa_nombres.values())))


Leyendo datos desde 'extraccion_completa_fireworks.csv'...
Iniciando desambiguación de autores con similitud de strings...
-> 550 variantes de nombres agrupadas en 456 autores únicos.


 SECCIÓN 3: CONSTRUCCIÓN DE CADA RED 

In [7]:
def guardar_csv(nombre_archivo, cabeceras, filas):
    """Función de ayuda para guardar archivos CSV."""
    with open(nombre_archivo, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(cabeceras)
        writer.writerows(filas)
    print(f"-> Archivo '{nombre_archivo}' guardado con éxito.")

In [8]:
def guardar_csv(nombre_archivo, cabeceras, filas):
    """Función de ayuda para guardar archivos CSV."""
    with open(nombre_archivo, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(cabeceras)
        writer.writerows(filas)
    print(f"-> Archivo '{nombre_archivo}' guardado con éxito.")

# --- 1. Red de Coautoría (Ponderada) ---
print("\n[1/4] Construyendo Red de Coautoría...")
aristas_coautoria = []
for art in articulos:
    autores_del_articulo = sorted(list(set([mapa_nombres.get(a) for a in art['autores'] if mapa_nombres.get(a) is not None])))
    if len(autores_del_articulo) > 1:
        aristas_coautoria.extend(list(combinations(autores_del_articulo, 2)))

aristas_ponderadas = Counter(aristas_coautoria)
filas_aristas_coautoria = [[src, tgt, 'Undirected', weight] for (src, tgt), weight in aristas_ponderadas.items()]
filas_nodos_autores = [[autor, autor] for autor in autores_canonicos_nodos]

guardar_csv('1_red_coautoria_nodos.csv', ['Id', 'Label'], filas_nodos_autores)
guardar_csv('1_red_coautoria_aristas.csv', ['Source', 'Target', 'Type', 'Weight'], filas_aristas_coautoria)


# --- 2. Red de Afiliación (Autor-Afiliación) ---
print("\n[2/4] Construyendo Red de Afiliación (Bipartita)...")
aristas_afiliacion = []
nodos_afiliaciones = set()
for art in articulos:
    autores_del_articulo = set([mapa_nombres.get(a) for a in art['autores'] if mapa_nombres.get(a) is not None])
    for autor in autores_del_articulo:
        for afiliacion in art['afiliaciones']:
            aristas_afiliacion.append([autor, afiliacion])
            nodos_afiliaciones.add(afiliacion)

# Para Gephi, en redes bipartitas, es útil tener una columna que distinga el tipo de nodo
filas_nodos_afiliacion = [[autor, autor, 'Autor'] for autor in autores_canonicos_nodos]
filas_nodos_afiliacion.extend([[afil, afil, 'Afiliacion'] for afil in sorted(list(nodos_afiliaciones))])
guardar_csv('2_red_afiliacion_nodos.csv', ['Id', 'Label', 'TipoNodo'], filas_nodos_afiliacion)
guardar_csv('2_red_afiliacion_aristas.csv', ['Source', 'Target'], aristas_afiliacion)


# --- 3. Red de Co-ocurrencia de Palabras Clave ---
print("\n[3/4] Construyendo Red de Co-ocurrencia de Palabras Clave...")
aristas_keywords = []
nodos_keywords = set()
for art in articulos:
    if len(art['palabras_clave']) > 1:
        aristas_keywords.extend(list(combinations(sorted(art['palabras_clave']), 2)))
    for kw in art['palabras_clave']:
        nodos_keywords.add(kw)

aristas_ponderadas_kw = Counter(aristas_keywords)
filas_aristas_kw = [[src, tgt, 'Undirected', weight] for (src, tgt), weight in aristas_ponderadas_kw.items()]
filas_nodos_kw = [[kw, kw] for kw in sorted(list(nodos_keywords))]

guardar_csv('3_red_keywords_nodos.csv', ['Id', 'Label'], filas_nodos_kw)
guardar_csv('3_red_keywords_aristas.csv', ['Source', 'Target', 'Type', 'Weight'], filas_aristas_kw)


# --- 4. Red Temática (Autor-Temática) ---
print("\n[4/4] Construyendo Red Temática (Bipartita)...")
aristas_tematica = []
nodos_tematicas = set()
for art in articulos:
    if art['tematica']:
        autores_del_articulo = set([mapa_nombres.get(a) for a in art['autores'] if mapa_nombres.get(a) is not None])
        nodos_tematicas.add(art['tematica'])
        for autor in autores_del_articulo:
            aristas_tematica.append([autor, art['tematica']])

filas_nodos_tematica = [[autor, autor, 'Autor'] for autor in autores_canonicos_nodos]
filas_nodos_tematica.extend([[tema, tema, 'Tematica'] for tema in sorted(list(nodos_tematicas))])

guardar_csv('4_red_tematica_nodos.csv', ['Id', 'Label', 'TipoNodo'], filas_nodos_tematica)
guardar_csv('4_red_tematica_aristas.csv', ['Source', 'Target'], aristas_tematica)


print("\n¡Proceso de generación de todas las redes completado!")


[1/4] Construyendo Red de Coautoría...
-> Archivo '1_red_coautoria_nodos.csv' guardado con éxito.
-> Archivo '1_red_coautoria_aristas.csv' guardado con éxito.

[2/4] Construyendo Red de Afiliación (Bipartita)...
-> Archivo '2_red_afiliacion_nodos.csv' guardado con éxito.
-> Archivo '2_red_afiliacion_aristas.csv' guardado con éxito.

[3/4] Construyendo Red de Co-ocurrencia de Palabras Clave...
-> Archivo '3_red_keywords_nodos.csv' guardado con éxito.
-> Archivo '3_red_keywords_aristas.csv' guardado con éxito.

[4/4] Construyendo Red Temática (Bipartita)...
-> Archivo '4_red_tematica_nodos.csv' guardado con éxito.
-> Archivo '4_red_tematica_aristas.csv' guardado con éxito.

¡Proceso de generación de todas las redes completado!
